# Final model: configuration decisions, LOG ablation and final seeds

This notebook fixes the final configuration from the earlier ablations with the decision rule (|z| ≥ 2.5; σ floors 0.003 m RMSE, 0.2 mIoU, 1 % AbsRel), runs the LOG ablation and trains the final seeds.

| Step | What | Output |
|---|---|---|
| 1 | configuration from notebooks 04–07, decided by the rule | `results/final/config_final.json`, `decisions.json` |
| 2 | LOG: linear vs log-space depth output (SwinMTL Tab. IV), seed 42, NYUv2 and Cityscapes | `ckpt/final/<dataset>_log_s42` |
| 3 | final seeds 43 and 44 on both datasets | `ckpt/final/<dataset>_base_s{42,43,44}` |
| 4 | comparison with SwinMTL, DINOv2 and DPT (mean ± std over 3 seeds) | `results/final/`; the up-to-date tables, with the NYU final model, are in notebook 03, 'Final model vs the three papers' |

**Final model actually used:** Cityscapes = the runs of this notebook. NYUv2 = the notebook 06 8-block arm (`ckpt/seeds/nc8_s{42,43,44}`: no frame-consistent PE, no photometric RNG fix), by the user's decision of 2026-11-09 (option A), because frame-consistent PE hurt NYU. The NYU runs of this notebook (`nyu_base_s*`) are superseded.


## Setup

***NOTE:*** The DINOv2 and DPT code is used straight from the local repositories `./dinov2` and `./DPT`.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "experiments":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [2]:
from mtl_final import *
from repro_common import PAPER     # the papers' reference numbers, checked against the PDFs

configure(results_root=ROOT / "results")
print(f"Device: {device}, GPU: {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'none'}")

COMPARE_DIR = CFG["results_root"] / "compare"
COMPARE_DIR.mkdir(parents=True, exist_ok=True)


def show_table(rows: List[dict], name: str, highlight: bool = False) -> pd.DataFrame:
    """Display a comparison table and save it to results/compare/<name>.csv.

    Args:
        rows: one dict per model, with a "model" key.
        name: file name.
        highlight: highlight my rows.
    """
    df = pd.DataFrame(rows).set_index("model")
    df.to_csv(COMPARE_DIR / f"{name}.csv")

    def fmt(value):
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return "—"
        if isinstance(value, float):
            return f"{value:.4g}"
        return value

    styled = df.style.format(fmt)
    if highlight:
        highlight_style = "font-weight: bold; background-color: rgba(255,200,0,.25)"
        styled = styled.apply(lambda row: [highlight_style if row.name.startswith("Mine") else "" for _ in row], axis=1)

    display(styled)
    return df


sigma: {'rmse': 0.003, 'miou': 0.2, 'abs_rel_cv': 0.01, 'source': 'notebook 06 (raw: rmse=0.001669, miou=0.1389, abs_rel_cv=0.005997)'}
Device: cuda, GPU: NVIDIA RTX A4000


## Helpers: σ, the decision rule, training and evaluation of a final-model run

In [3]:
# The decision rule, sigma, log_bias, setup_final, train_eval and load_eval are defined in mtl_final.py
print("sigma:", SIGMA)


sigma: {'rmse': 0.003, 'miou': 0.2, 'abs_rel_cv': 0.01, 'source': 'notebook 06 (raw: rmse=0.001669, miou=0.1389, abs_rel_cv=0.005997)'}


## Step 1: configuration from the finished ablations
Reads:
- E2 vs B (critic);
- the 3 + 3 seeds of notebook 06 (4 vs 8 blocks, compared by their means);
- AUG-A…D of notebook 07 against the 8-block baseline (re-based from E2 on 2026-11-05).

Missing results leave the default setting in place.

In [4]:
CONFIG = dict(blocks=4, use_critic=False, frame_pe=True, photo_rng_fix=False, depth_param="linear",
              nyu_scale_aug=(1.0, 1.5), nyu_crop=None, nyu_rotate_deg=0.0)

R_B = load_eval(ROOT / "results", "nyu_partial_ft")
R_E2 = load_eval(ROOT / "results" / "nocritic", "nyu_partial_ft")

# critic (E2): kept off unless removing it was harmful
if R_B and R_E2:
    j = judge("critic off (E2 vs B)", "nyu", R_B, R_E2)
    CONFIG["use_critic"] = bool(j["harm"])
    record("critic", "on" if CONFIG["use_critic"] else "off", "kept off unless removing it is harmful (1.8x cheaper)", [j])
else:
    record("critic", "off", "E2 / B results missing: default off")

# trainable blocks (notebook 06, means of 3 seeds per arm)
seed_csv = ROOT / "results" / "compare" / "seeds_nyu_per_run.csv"
if seed_csv.exists():
    df = pd.read_csv(seed_csv)
    a, b = df[df.arm == "4 blocks"], df[df.arm == "8 blocks"]
    fake = lambda d: {"nyu_depth": {"rmse": d["RMSE"].mean()}, "nyu_seg": {"mIoU": d["mIoU"].mean()}}
    j = judge(f"8 vs 4 blocks (means of {len(b)} vs {len(a)} seeds)", "nyu", fake(a), fake(b))
    CONFIG["blocks"] = 8 if (j["better"] and not j["harm"]) else 4
    record("trainable blocks", CONFIG["blocks"], "8 only if better by >= 2.5 sigma without harm, else the cheaper 4", [j])
else:
    record("trainable blocks", 4, "notebook 06 results missing: default 4")

# augmentations (notebook 07) vs the 8-block baseline (user decision 2026-11-05: re-based from E2 to 8 blocks)
R_8 = load_eval(ROOT / "results" / "seeds" / "nc8_s42", "nyu_partial_ft8")
AUG_SETTINGS = {
    "AUG-A": dict(nyu_scale_aug=(1.0, 1.0), nyu_crop=(448, 574)),
    "AUG-B": dict(nyu_scale_aug=(0.5, 2.0)),
    "AUG-C": dict(nyu_rotate_deg=2.5),
    "AUG-D": dict(photo_rng_fix=True),
}
aug_j = {}
for aug_id in AUG_SETTINGS:
    R = load_eval(ROOT / "results" / "aug" / aug_id.lower().replace("-", "_"), "nyu_partial_ft8")
    if R and R_8:
        aug_j[aug_id] = judge(f"{aug_id} vs 8-block baseline", "nyu", R_8, R)

# rescale: AUG-A and AUG-B are exclusive
rescale = [k for k in ("AUG-A", "AUG-B") if k in aug_j and aug_j[k]["better"] and not aug_j[k]["harm"]]
if rescale:
    best = max(rescale, key=lambda k: aug_j[k]["z_depth"] + aug_j[k]["z_seg"])
    CONFIG.update(AUG_SETTINGS[best])
    record("rescale / crop (AUG-A vs AUG-B)", best, "adopted by the rule; the larger summed z wins if both pass",
           [aug_j[k] for k in ("AUG-A", "AUG-B") if k in aug_j])
else:
    record("rescale / crop (AUG-A vs AUG-B)", "unchanged [1, 1.5], crop 476x630", "neither passed the rule (or missing)",
           [aug_j[k] for k in ("AUG-A", "AUG-B") if k in aug_j])

if "AUG-C" in aug_j and aug_j["AUG-C"]["better"] and not aug_j["AUG-C"]["harm"]:
    CONFIG.update(AUG_SETTINGS["AUG-C"])
record("rotation (AUG-C)", CONFIG["nyu_rotate_deg"], "adopted only if it passes the rule", [aug_j["AUG-C"]] if "AUG-C" in aug_j else [])

if "AUG-D" in aug_j:
    CONFIG["photo_rng_fix"] = not aug_j["AUG-D"]["harm"]
    record("photometric RNG fix (AUG-D)", CONFIG["photo_rng_fix"], "bug fix: adopted unless harmful", [aug_j["AUG-D"]])
else:
    CONFIG["photo_rng_fix"] = True
    record("photometric RNG fix (AUG-D)", True, "AUG-D missing: adopted as the intended behaviour")

record("frame-consistent position embeddings", True, "user decision (2026-11-04): both datasets; NYU ablation in notebook 10")
save_json(CONFIG, FINAL_DIR / "config_step1.json")
display(pd.DataFrame([dict(j) for j in aug_j.values()]).style.format(precision=4) if aug_j else "no AUG results")
print("configuration after step 1:", CONFIG)


[decision] critic: off  (kept off unless removing it is harmful (1.8x cheaper))
[decision] trainable blocks: 8  (8 only if better by >= 2.5 sigma without harm, else the cheaper 4)
[decision] rescale / crop (AUG-A vs AUG-B): unchanged [1, 1.5], crop 476x630  (neither passed the rule (or missing))
[decision] rotation (AUG-C): 0.0  (adopted only if it passes the rule)
[decision] photometric RNG fix (AUG-D): True  (bug fix: adopted unless harmful)
[decision] frame-consistent position embeddings: True  (user decision (2026-11-04): both datasets; NYU ablation in notebook 10)


,change,dataset,base_depth,new_depth,base_mIoU,new_mIoU,z_depth,z_seg,better,harm,kind
0,AUG-A vs 8-block baseline,nyu,0.4205,0.4365,58.2490,56.4518,-5.3418,-8.9861,False,True,standard
1,AUG-B vs 8-block baseline,nyu,0.4205,0.4244,58.2490,58.6394,-1.2934,1.9516,False,False,standard
2,AUG-C vs 8-block baseline,nyu,0.4205,0.4183,58.2490,58.1729,0.7300,-0.3808,False,False,standard
3,AUG-D vs 8-block baseline,nyu,0.4205,0.4205,58.2490,58.1319,0.0114,-0.5857,False,False,standard


configuration after step 1: {'blocks': 8, 'use_critic': False, 'frame_pe': True, 'photo_rng_fix': True, 'depth_param': 'linear', 'nyu_scale_aug': (1.0, 1.5), 'nyu_crop': None, 'nyu_rotate_deg': 0.0}


## Step 2: log-depth ablation (LOG), seed 42, both datasets
`base` = the step-1 configuration with the linear output; `log` = the same with SwinMTL's log-space output. NYUv2 first (≈ 3 h each), then Cityscapes (≈ 8 h each). LOG is adopted if, on at least one dataset, it passes the rule, and it is not harmful on either.

In [5]:
CFG_LIN = dict(CONFIG, depth_param="linear")
CFG_LOG = dict(CONFIG, depth_param="log")

R_LOG = {}
for dataset in ("nyu", "cityscapes"):
    R_LOG[(dataset, "base")] = train_eval(dataset, 42, "base", CFG_LIN)
    R_LOG[(dataset, "log")] = train_eval(dataset, 42, "log", CFG_LOG)

log_j = [judge("log-space depth output vs linear", d, R_LOG[(d, "base")], R_LOG[(d, "log")]) for d in ("nyu", "cityscapes")]
adopt_log = any(j["better"] for j in log_j) and not any(j["harm"] for j in log_j)
CONFIG["depth_param"] = "log" if adopt_log else "linear"
FINAL_TAG = "log" if adopt_log else "base"
record("depth output (LOG)", CONFIG["depth_param"], "log adopted if better on one dataset and harmful on none", log_j)
save_json(CONFIG, FINAL_DIR / "config_final.json")

# combined check (user decision 2026-11-05, "option 2" added to option 1, no extra run):
# the final NYU configuration (seed 42) vs the 8-block baseline (seed 42, nothing else adopted).
# On NYU it differs from the baseline by the adopted augmentations + frame-consistent PE (+ LOG if adopted).
if R_8:
    combo = judge("combined final config vs 8-block baseline (NYU, seed 42)", "nyu", R_8, R_LOG[("nyu", FINAL_TAG)], kind="check")
    record("combined check", "ok" if not combo["harm"] else "WARNING: combined config worse than the baseline",
           "no decision: confirms that the adopted changes still help together", [combo])
display(pd.DataFrame(log_j).style.format(precision=4))
print("final configuration:", CONFIG, "| seed 42 run:", FINAL_TAG)


=== nyu base seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base_s42
nyu_final: already trained
=== nyu log seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_log_s42
nyu_final: already trained
=== cityscapes base seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/cityscapes_base_s42
cityscapes_final: already trained
=== cityscapes log seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/cityscapes_log_s42
cityscapes_final: already trained
[decision] depth output (LOG): linear  (log adopted if better on one dataset and harmful on none)
[decision] combined check: WARNING: combined config worse than the baseline  (no decision: confirms that the adopted changes still help together)


,change,dataset,base_depth,new_depth,base_mIoU,new_mIoU,z_depth,z_seg,better,harm,kind
0,log-space depth output vs linear,nyu,0.4271,0.4264,57.3942,57.6985,0.2301,1.5213,False,False,standard
1,log-space depth output vs linear,cityscapes,0.1059,0.1089,77.9468,77.7630,-2.9263,-0.9192,False,True,standard


final configuration: {'blocks': 8, 'use_critic': False, 'frame_pe': True, 'photo_rng_fix': True, 'depth_param': 'linear', 'nyu_scale_aug': (1.0, 1.5), 'nyu_crop': None, 'nyu_rotate_deg': 0.0} | seed 42 run: base


## Step 3: final model, seeds 43 and 44
Seed 42 is the winning run of step 2. NYUv2 first, then Cityscapes.

In [6]:
FINAL_SEEDS = [42, 43, 44]
R_FINAL = {d: {42: R_LOG[(d, FINAL_TAG)]} for d in ("nyu", "cityscapes")}
for dataset in ("nyu", "cityscapes"):
    for seed in FINAL_SEEDS[1:]:
        R_FINAL[dataset][seed] = train_eval(dataset, seed, FINAL_TAG, CONFIG)
save_json({d: {str(s): r for s, r in v.items()} for d, v in R_FINAL.items()}, FINAL_DIR / "final_eval_all_seeds.json")


=== nyu base seed 43: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base_s43
nyu_final: already trained
=== nyu base seed 44: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base_s44
nyu_final: already trained
=== cityscapes base seed 43: /home/chrisjf/Documents/ml4cv/ckpt/final/cityscapes_base_s43
cityscapes_final: already trained
=== cityscapes base seed 44: /home/chrisjf/Documents/ml4cv/ckpt/final/cityscapes_base_s44
cityscapes_final: already trained


## Step 4: evaluation, mean ± std over 3 seeds, comparison with the papers
Same protocols and tables as notebook 03, 'One-to-one comparisons': SwinMTL Tab. I / II / III, DINOv2 Tab. 10 / 11, DPT Tab. 2. The final model is shown as mean ± std. The best and worst seed are listed too, so it is visible whether *every* seed beats a paper number (03 §9 protocol). The sliding-window rows (TTA) are supplementary.

In [7]:
def agg(dataset: str, key: str, metric: str) -> Tuple[float, float, float, float]:
    vals = np.array([R_FINAL[dataset][s][key][metric] for s in FINAL_SEEDS if key in R_FINAL[dataset][s]])
    return float(vals.mean()), float(vals.std(ddof=1)) if len(vals) > 1 else float("nan"), float(vals.min()), float(vals.max())


def pm(dataset, key, metric):
    m, s, lo, hi = agg(dataset, key, metric)
    return f"{m:.4g} ± {s:.2g}"


FINAL_LABEL = "Final (mean ± std, 3 seeds)"
NYU_LABEL = "NYU run of this notebook, frame PE (superseded: NYU final = notebook 06 8-block arm)"
summary_rows = []
for dataset, protocols in [("cityscapes", [("cs_depth_512x1024", ["abs_rel", "sq_rel", "rmse", "rmse_log", "d1"]), ("cs_depth_full", ["rmse"]),
                                           ("cs_seg", ["mIoU"]), ("cs_seg_ms", ["mIoU"]), ("cs_seg_swinmtl", ["mIoU"]),
                                           ("cs_depth_512x1024_slide", ["abs_rel", "rmse"]), ("cs_seg_slide", ["mIoU"])]),
                           ("nyu", [("nyu_depth", ["rmse", "abs_rel", "d1", "d2", "d3", "log10"]), ("nyu_depth_flip", ["rmse"]),
                                    ("nyu_seg", ["mIoU"]), ("nyu_seg_swinmtl", ["mIoU"]), ("nyu_depth_slide", ["rmse"]), ("nyu_seg_slide", ["mIoU"])])]:
    for key, metrics in protocols:
        for metric in metrics:
            m, s, lo, hi = agg(dataset, key, metric)
            summary_rows.append(dict(dataset=dataset, protocol=key, metric=metric, mean=m, std=s, min=lo, max=hi,
                                     **{f"seed {sd}": R_FINAL[dataset][sd][key][metric] for sd in FINAL_SEEDS}))
for dataset in ("cityscapes", "nyu"):
    lb = np.array([R_FINAL[dataset][s]["log_bias"] for s in FINAL_SEEDS])
    summary_rows.append(dict(dataset=dataset, protocol="single pass", metric="log_bias", mean=lb.mean(), std=lb.std(ddof=1), min=lb.min(), max=lb.max(),
                             **{f"seed {sd}": R_FINAL[dataset][sd]["log_bias"] for sd in FINAL_SEEDS}))
final_summary = pd.DataFrame(summary_rows)
final_summary.to_csv(FINAL_DIR / "final_summary.csv", index=False)
display(final_summary.style.format(precision=4))


,dataset,protocol,metric,mean,std,min,max,seed 42,seed 43,seed 44
0,cityscapes,cs_depth_512x1024,abs_rel,0.1082,0.0021,0.1059,0.1100,0.1059,0.1100,0.1089
1,cityscapes,cs_depth_512x1024,sq_rel,0.8228,0.0027,0.8199,0.8252,0.8233,0.8252,0.8199
2,cityscapes,cs_depth_512x1024,rmse,5.0244,0.0192,5.0096,5.0462,5.0462,5.0175,5.0096
3,cityscapes,cs_depth_512x1024,rmse_log,0.1461,0.0009,0.1450,0.1468,0.1450,0.1468,0.1464
4,cityscapes,cs_depth_512x1024,d1,0.9140,0.0003,0.9136,0.9143,0.9136,0.9143,0.9142
5,cityscapes,cs_depth_full,rmse,5.0591,0.0189,5.0449,5.0806,5.0806,5.0518,5.0449
6,cityscapes,cs_seg,mIoU,77.7365,0.1863,77.5922,77.9468,77.9468,77.5922,77.6704
7,cityscapes,cs_seg_ms,mIoU,79.7864,0.1263,79.6545,79.9063,79.9063,79.6545,79.7984
8,cityscapes,cs_seg_swinmtl,mIoU,65.2273,0.0170,65.2078,65.2387,65.2078,65.2354,65.2387
9,cityscapes,cs_depth_512x1024_slide,abs_rel,0.0828,0.0001,0.0827,0.0829,0.0827,0.0828,0.0829


### Final vs SwinMTL (Tab. I, II, III)

In [8]:
# Tab. I: Cityscapes depth at 512x1024
rows = [dict(model=model, source="paper", **values) for model, values in PAPER["swinmtl_tab1"].items() if model == "SwinMTL" or not model.endswith("*")]
rows.append(dict(model=FINAL_LABEL, source="measured", res="512x1024",
                 **{k: pm("cityscapes", "cs_depth_512x1024", k) for k in ["abs_rel", "sq_rel", "rmse", "rmse_log", "d1", "d2", "d3"]}))
rows.append(dict(model=FINAL_LABEL + " +slide (TTA)", source="measured, TTA", res="512x1024",
                 **{k: pm("cityscapes", "cs_depth_512x1024_slide", k) for k in ["abs_rel", "sq_rel", "rmse", "rmse_log", "d1", "d2", "d3"]}))
print("SwinMTL Tab. I: Cityscapes depth")
show_table(rows, "final_swinmtl_tab1")

# Tab. II: Cityscapes MTL at 1024x2048
rows = [{"model": m, "source": "paper", "rmse": r, "mIoU (paper: as reported; ours: standard)": i, "mIoU (ours: SwinMTL per-image)": None} for m, (r, i) in PAPER["swinmtl_tab2"].items()]
rows.append({"model": FINAL_LABEL, "source": "measured", "rmse": pm("cityscapes", "cs_depth_full", "rmse"),
             "mIoU (paper: as reported; ours: standard)": pm("cityscapes", "cs_seg", "mIoU"), "mIoU (ours: SwinMTL per-image)": pm("cityscapes", "cs_seg_swinmtl", "mIoU")})
print("SwinMTL Tab. II: Cityscapes MTL (mIoU: papers as reported, ours standard; SwinMTL per-image shown for reference)")
show_table(rows, "final_swinmtl_tab2")

# Tab. III: NYUv2 MTL
rows = [{"model": m, "source": "paper", "rmse": r, "mIoU (paper: as reported; ours: standard)": i, "mIoU (ours: SwinMTL per-image)": None} for m, (r, i) in PAPER["swinmtl_tab3"].items()]
rows.append({"model": NYU_LABEL, "source": "measured", "rmse": pm("nyu", "nyu_depth", "rmse"),
             "mIoU (paper: as reported; ours: standard)": pm("nyu", "nyu_seg", "mIoU"), "mIoU (ours: SwinMTL per-image)": pm("nyu", "nyu_seg_swinmtl", "mIoU")})
rows.append({"model": NYU_LABEL + " +slide (TTA)", "source": "measured, TTA", "rmse": pm("nyu", "nyu_depth_slide", "rmse"),
             "mIoU (paper: as reported; ours: standard)": pm("nyu", "nyu_seg_slide", "mIoU"), "mIoU (ours: SwinMTL per-image)": None})
print("SwinMTL Tab. III: NYUv2 MTL (NYU rows: superseded frame-PE run; the NYU final model is in notebook 03)")
show_table(rows, "final_swinmtl_tab3")


SwinMTL Tab. I: Cityscapes depth


,source,res,abs_rel,sq_rel,rmse,rmse_log,d1,d2,d3
model,,,,,,,,,
GLNet,paper,1024x2048,0.111,—,6.437,0.182,0.868,0.961,0.983
SwinMTL,paper,512x1024,0.089,1.051,5.481,0.139,0.921,0.976,0.99
"Final (mean ± std, 3 seeds)",measured,512x1024,0.1082 ± 0.0021,0.8228 ± 0.0027,5.024 ± 0.019,0.1461 ± 0.00094,0.914 ± 0.00034,0.9815 ± 7.6e-05,0.9939 ± 6.9e-05
"Final (mean ± std, 3 seeds) +slide (TTA)","measured, TTA",512x1024,0.08281 ± 7.5e-05,0.6757 ± 0.0055,4.65 ± 0.0095,0.1309 ± 3.2e-05,0.9274 ± 0.00021,0.9836 ± 1e-05,0.9947 ± 3.8e-05


SwinMTL Tab. II: Cityscapes MTL (mIoU: papers as reported, ours standard; SwinMTL per-image shown for reference)


,source,rmse,mIoU (paper: as reported; ours: standard),mIoU (ours: SwinMTL per-image)
model,,,,
Hybridnet,paper,12.09,66.61,—
MGMNet,paper,8.3,55.7,—
MTL,paper,6.797,70.43,—
PAD-Net,paper,6.777,70.23,—
3-ways,paper,6.528,75,—
[44],paper,6.649,74.95,—
SwinMTL,paper,6.352,76.41,—
"Final (mean ± std, 3 seeds)",measured,5.059 ± 0.019,77.74 ± 0.19,65.23 ± 0.017


SwinMTL Tab. III: NYUv2 MTL (NYU rows: superseded frame-PE run; the NYU final model is in notebook 03)


,source,rmse,mIoU (paper: as reported; ours: standard),mIoU (ours: SwinMTL per-image)
model,,,,
Cross-Stitch,paper,0.629,36.34,—
MTL,paper,0.638,39.44,—
PAD-Net,paper,0.627,36.61,—
3-ways,paper,0.622,39.47,—
[44],paper,0.604,38.93,—
MTAN,paper,0.5906,40.01,—
MTI-Net,paper,0.5365,45.97,—
ATRC,paper,0.5363,46.33,—
InvPT,paper,0.5183,53.56,—


,source,rmse,mIoU (paper: as reported; ours: standard),mIoU (ours: SwinMTL per-image)
model,,,,
Cross-Stitch,paper,0.629,36.34,NaN
MTL,paper,0.638,39.44,NaN
PAD-Net,paper,0.627,36.61,NaN
3-ways,paper,0.622,39.47,NaN
[44],paper,0.604,38.93,NaN
MTAN,paper,0.5906,40.01,NaN
MTI-Net,paper,0.5365,45.97,NaN
ATRC,paper,0.5363,46.33,NaN
InvPT,paper,0.5183,53.56,NaN


### Final vs DINOv2 ViT-S/14 (Tab. 10, 11) and DPT (Tab. 2)

In [9]:
dd, ds_ = PAPER["dinov2_tab11_vits14"], PAPER["dinov2_tab10_vits14"]
rows = [{"model": f"DINOv2 ViT-S/14 {s}", "source": "paper (Tab. 11)", "NYU RMSE": dd["nyu"][i]} for i, s in enumerate(["lin. 1", "lin. 4", "DPT"])]
rows.append({"model": NYU_LABEL, "source": "measured", "NYU RMSE": pm("nyu", "nyu_depth", "rmse"),
             "NYU RMSE (flip)": pm("nyu", "nyu_depth_flip", "rmse"), "NYU RMSE (slide, TTA)": pm("nyu", "nyu_depth_slide", "rmse")})
print("DINOv2 Tab. 11: NYUv2 depth (RMSE)")
show_table(rows, "final_dinov2_nyu_depth")

rows = [{"model": "DINOv2 ViT-S/14 (frozen)", "source": "paper (Tab. 10)", "CS mIoU (single-scale / lin.)": ds_["cityscapes"][0], "CS mIoU (+ms, TTA)": ds_["cityscapes"][1]}]
rows.append({"model": FINAL_LABEL, "source": "measured", "CS mIoU (single-scale / lin.)": pm("cityscapes", "cs_seg", "mIoU"),
             "CS mIoU (+ms, TTA)": pm("cityscapes", "cs_seg_ms", "mIoU"), "CS mIoU (slide, TTA)": pm("cityscapes", "cs_seg_slide", "mIoU")})
print("DINOv2 Tab. 10: Cityscapes segmentation (mIoU, 19 classes)")
show_table(rows, "final_dinov2_cs_seg")

rows = [dict(model=m, source="paper (Tab. 2)", **v) for m, v in PAPER["dpt_tab2"].items()]
rows.append(dict(model=NYU_LABEL, source="measured", **{k: pm("nyu", "nyu_depth", k) for k in ["d1", "d2", "d3", "abs_rel", "rmse", "log10"]}))
rows.append(dict(model=NYU_LABEL + " +flip (TTA)", source="measured, TTA", **{k: pm("nyu", "nyu_depth_flip", k) for k in ["d1", "d2", "d3", "abs_rel", "rmse", "log10"]}))
print("DPT Tab. 2: NYUv2 depth")
show_table(rows, "final_dpt_nyu")


DINOv2 Tab. 11: NYUv2 depth (RMSE)


,source,NYU RMSE,NYU RMSE (flip),"NYU RMSE (slide, TTA)"
model,,,,
DINOv2 ViT-S/14 lin. 1,paper (Tab. 11),0.449,—,—
DINOv2 ViT-S/14 lin. 4,paper (Tab. 11),0.417,—,—
DINOv2 ViT-S/14 DPT,paper (Tab. 11),0.356,—,—
"NYU run of this notebook, frame PE (superseded: NYU final = notebook 06 8-block arm)",measured,0.427 ± 0.0003,0.4209 ± 0.00062,0.417 ± 0.0007


DINOv2 Tab. 10: Cityscapes segmentation (mIoU, 19 classes)


,source,CS mIoU (single-scale / lin.),"CS mIoU (+ms, TTA)","CS mIoU (slide, TTA)"
model,,,,
DINOv2 ViT-S/14 (frozen),paper (Tab. 10),66.6,77.1,—
"Final (mean ± std, 3 seeds)",measured,77.74 ± 0.19,79.79 ± 0.13,78.98 ± 0.19


DPT Tab. 2: NYUv2 depth


,source,d1,d2,d3,abs_rel,rmse,log10
model,,,,,,,
DORN,paper (Tab. 2),0.828,0.965,0.992,0.115,0.509,0.051
VNL,paper (Tab. 2),0.875,0.976,0.994,0.111,0.416,0.048
BTS,paper (Tab. 2),0.885,0.978,0.994,0.11,0.392,0.047
DPT-Hybrid,paper (Tab. 2),0.904,0.988,0.998,0.11,0.357,0.045
"NYU run of this notebook, frame PE (superseded: NYU final = notebook 06 8-block arm)",measured,0.8625 ± 0.00053,0.9822 ± 0.00064,0.9968 ± 6.6e-05,0.1216 ± 0.00035,0.427 ± 0.0003,0.05134 ± 0.00013
"NYU run of this notebook, frame PE (superseded: NYU final = notebook 06 8-block arm) +flip (TTA)","measured, TTA",0.8662 ± 0.00093,0.9829 ± 0.00086,0.9969 ± 0.00013,0.1199 ± 0.00024,0.4209 ± 0.00062,0.05066 ± 5.6e-05


,source,d1,d2,d3,abs_rel,rmse,log10
model,,,,,,,
DORN,paper (Tab. 2),0.828,0.965,0.992,0.115,0.509,0.051
VNL,paper (Tab. 2),0.875,0.976,0.994,0.111,0.416,0.048
BTS,paper (Tab. 2),0.885,0.978,0.994,0.11,0.392,0.047
DPT-Hybrid,paper (Tab. 2),0.904,0.988,0.998,0.11,0.357,0.045
"NYU run of this notebook, frame PE (superseded: NYU final = notebook 06 8-block arm)",measured,0.8625 ± 0.00053,0.9822 ± 0.00064,0.9968 ± 6.6e-05,0.1216 ± 0.00035,0.427 ± 0.0003,0.05134 ± 0.00013
"NYU run of this notebook, frame PE (superseded: NYU final = notebook 06 8-block arm) +flip (TTA)","measured, TTA",0.8662 ± 0.00093,0.9829 ± 0.00086,0.9969 ± 0.00013,0.1199 ± 0.00024,0.4209 ± 0.00062,0.05066 ± 5.6e-05


### Targets (single pass): does the mean, and does every seed, beat each paper number?

In [10]:
from repro_common import FINAL_TARGETS   # defined once in repro_common.py
TARGETS = [((("[superseded NYU run of this notebook] " + t[0]) if t[1] == "nyu" else t[0]),) + tuple(t[1:]) for t in FINAL_TARGETS]
rows = []
for label, dataset, key, metric, paper, lower in TARGETS:
    vals = np.array([R_FINAL[dataset][s][key][metric] for s in FINAL_SEEDS])
    beats = (vals < paper) if lower else (vals > paper)
    rows.append(dict(target=label, paper=paper, mean=vals.mean(), std=vals.std(ddof=1),
                     mean_beats=bool((vals.mean() < paper) if lower else (vals.mean() > paper)),
                     seeds_beating=f"{int(beats.sum())}/{len(vals)}"))
targets = pd.DataFrame(rows).set_index("target")
targets.to_csv(FINAL_DIR / "targets.csv")
display(targets.style.format(precision=4))


,paper,mean,std,mean_beats,seeds_beating
target,,,,,
CS AbsRel vs SwinMTL,0.0890,0.1082,0.0021,False,0/3
CS SqRel vs SwinMTL,1.0510,0.8228,0.0027,True,3/3
CS RMSE (512) vs SwinMTL,5.4810,5.0244,0.0192,True,3/3
CS RMSElog vs SwinMTL,0.1390,0.1461,0.0009,False,0/3
CS δ1 vs SwinMTL,0.9210,0.9140,0.0003,False,0/3
CS RMSE (full) vs SwinMTL,6.3520,5.0591,0.0189,True,3/3
CS mIoU (standard) vs SwinMTL,76.4100,77.7365,0.1863,True,3/3
CS mIoU (SwinMTL per-image) vs SwinMTL,76.4100,65.2273,0.0170,False,0/3
"CS mIoU vs DINOv2 (frozen, lin.)",66.6000,77.7365,0.1863,True,3/3
